# Does a wider network help a DOOM agent, or just cost more?

The laptop version of this agent ([github.com/hammasbuilds/game-agents](https://github.com/hammasbuilds/game-agents)) is deliberately small: 219,476 parameters, the DQN trunk with the channels halved, because it had to train on a CPU. It reaches **9.80 kills per episode** on ViZDoom's `defend_the_center`, against 1.23 for a hand-written turn-and-shoot baseline.

That raises an obvious question the laptop cannot answer: was the small network a compromise, or was it enough? This notebook runs both arms on the same budget — same steps, same environments, same seeds — and changes only the width.

**What is being compared**

| arm | channels | parameters | device |
|---|---|---|---|
| control | 16/32/32, hidden 256 | ~219k | GPU |
| wide | 64/128/128, hidden 1024 | ~13M | GPU |

Both arms run on the GPU, so the comparison is about capacity and not about hardware. Note what that implies: ViZDoom's rollout is CPU work, and this box has four cores, so **the GPU does not make the wide arm slower per step by much — and does not make either arm fast**. The bottleneck is the game, not the gradient.

In [ ]:
# A quota-exhausted Kaggle session quietly gives you a CPU and runs to
# completion looking perfectly healthy. Fail here instead, loudly.
import torch, os, multiprocessing

assert torch.cuda.is_available(), "no GPU in this session: set Accelerator to GPU, or wait for quota"
print("gpu   :", torch.cuda.get_device_name(0))
print("torch :", torch.__version__)
print("cores :", multiprocessing.cpu_count())

In [ ]:
!pip -q install vizdoom==1.3.2
!git clone -q https://github.com/hammasbuilds/game-agents.git /kaggle/working/game-agents
%cd /kaggle/working/game-agents/projects/01-doom-pixel-rl
import vizdoom, pathlib
print("vizdoom", vizdoom.__version__)
# ViZDoom ships Freedoom, so there is no commercial DOOM data to find.
print(sorted(p.name for p in pathlib.Path(vizdoom.__path__[0]).glob('*.wad')))

## Train both arms

`--envs 4` matches the core count: every environment is a separate DOOM process, and oversubscribing the cores slows the rollout down without buying any more experience. 1.2M steps each keeps the pair inside a single session.

In [ ]:
STEPS = 1_200_000
!python train.py --scenario defend_the_center --steps {STEPS} --envs 4 \
    --width 1 --device cuda --seed 0 --out runs/control

In [ ]:
!python train.py --scenario defend_the_center --steps {STEPS} --envs 4 \
    --width 4 --device cuda --seed 0 --out runs/wide

## Score both on maps neither of them trained on

Evaluation seeds start at 900000; training seeds are 0 and 1000·i, so no evaluated map is one either policy was fitted on. Both arms see the same thirty.

In [ ]:
!python evaluate.py --scenario defend_the_center --episodes 30 \
    --policy runs/control/policy.pt --out runs/control/eval.json
!python evaluate.py --scenario defend_the_center --episodes 30 \
    --policy runs/wide/policy.pt --out runs/wide/eval.json

In [ ]:
import json, pathlib, shutil

summary = {}
for arm in ("control", "wide"):
    base = pathlib.Path("runs") / arm
    ev = json.loads((base / "eval.json").read_text())
    tr = json.loads((base / "train_summary.json").read_text())
    policy = [r for r in ev["rows"] if r["agent"].startswith("ppo")]
    summary[arm] = {
        "params": tr["params"], "width": tr["width"], "device": tr["device"],
        "wall_seconds": tr["wall_seconds"], "steps": tr["steps"],
        "steps_per_second": round(tr["steps"] / tr["wall_seconds"], 1),
        "eval": {r["agent"]: {"mean": r["return_mean"], "sd": r["return_std"]}
                 for r in ev["rows"]},
        "best_policy_mean": max(r["return_mean"] for r in policy),
    }

print(f"{'arm':<9}{'params':>10}{'steps/s':>10}{'minutes':>9}{'kills/episode':>16}")
for arm, row in summary.items():
    print(f"{arm:<9}{row['params']:>10,}{row['steps_per_second']:>10}"
          f"{row['wall_seconds'] / 60:>9.0f}{row['best_policy_mean']:>16.2f}")

out = pathlib.Path("/kaggle/working")
(out / "width_comparison.json").write_text(json.dumps(summary, indent=1))
for arm in ("control", "wide"):
    for name in ("policy.pt", "history.json", "eval.json"):
        src = pathlib.Path("runs") / arm / name
        if src.exists():
            shutil.copy(src, out / f"{arm}_{name}")
print("\nwrote width_comparison.json and both policies to /kaggle/working")

## Reading the result

The honest thing to watch here is not just which arm scores higher but what the extra parameters cost. `defend_the_center` gives the agent three buttons and a room; if the wide arm does not clear the control arm by more than the spread between seeds, then the laptop-sized network was not a compromise at all, and the right place to spend compute is more steps rather than more weights.

The repository this notebook clones holds the CPU run it is being compared against, along with two other game-playing agents: a Crafter planner with no learned weights, and an agent that plays Brogue through screenshots and keystrokes.